In [ ]:
import torch
import pandas as pd
from tqdm import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification,
    pipeline
)


In [ ]:
############################################
# CONFIG
############################################
INPUT_CSV = "test.csv"
OUTPUT_CSV = "test_translated_scored.csv"
BATCH_SIZE = 16
MAX_INPUT_LEN = 512
MAX_NEW_TOKENS = 256

TRANSLATION_MODEL = "tencent/Hunyuan-MT-Chimera-7B"
SENTIMENT_MODEL = "nlptown/bert-base-multilingual-uncased-sentiment"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
############################################
# LOAD TRANSLATION MODEL
############################################
print("Loading Hunyuan-MT-Chimera-7B...")

mt_tokenizer = AutoTokenizer.from_pretrained(
    TRANSLATION_MODEL,
    trust_remote_code=True
)

mt_model = AutoModelForSeq2SeqLM.from_pretrained(
    TRANSLATION_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

In [ ]:
############################################
# LOAD SENTIMENT MODEL
############################################
print("Loading sentiment model...")

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=SENTIMENT_MODEL,
    tokenizer=SENTIMENT_MODEL,
    device=0 if DEVICE == "cuda" else -1
)

In [ ]:
############################################
# TRANSLATION FUNCTION
############################################
def translate_to_english(texts, src_lang):
    """
    texts: List[str]
    src_lang: language code (de, fr, es, ja, zh, en)
    """
    prompts = [
        f"<|{src_lang}|> <|en|> {text}" for text in texts
    ]

    inputs = mt_tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LEN
    ).to(mt_model.device)

    with torch.no_grad():
        outputs = mt_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=4
        )

    translations = mt_tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )

    return translations

In [ ]:
############################################
# SENTIMENT SCORING FUNCTION
############################################
def get_sentiment_scores(texts):
    """
    Returns integer scores from 1 to 5
    """
    results = sentiment_pipeline(texts, batch_size=32)

    scores = []
    for r in results:
        # label format: "1 star", "2 stars", ...
        score = int(r["label"].split()[0])
        scores.append(score)

    return scores

In [ ]:
############################################
# MAIN PIPELINE
############################################
def process_reviews(df):
    translated_reviews = []
    sentiment_scores = []

    for i in tqdm(range(0, len(df), BATCH_SIZE)):
        batch = df.iloc[i:i + BATCH_SIZE]

        texts = batch["review_body"].fillna("").tolist()
        langs = batch["language"].tolist()

        # Group by language inside the batch
        translations_map = {}

        for lang in set(langs):
            indices = [idx for idx, l in enumerate(langs) if l == lang]
            lang_texts = [texts[idx] for idx in indices]

            if lang == "en":
                translated = lang_texts
            else:
                translated = translate_to_english(lang_texts, lang)

            for idx, t in zip(indices, translated):
                translations_map[idx] = t

        # Restore original order
        ordered_translations = [translations_map[i] for i in range(len(batch))]

        # Sentiment scoring
        scores = get_sentiment_scores(ordered_translations)

        translated_reviews.extend(ordered_translations)
        sentiment_scores.extend(scores)

    return translated_reviews, sentiment_scores

In [ ]:
############################################
# RUN
############################################
if __name__ == "__main__":
    print("Reading CSV...")
    df = pd.read_csv(INPUT_CSV)

    print("Translating and scoring reviews...")
    translated_reviews, sentiment_scores = process_reviews(df)

    df["review_body_en"] = translated_reviews
    df["sentiment_score_1_to_5"] = sentiment_scores

    print(f"Saving output to {OUTPUT_CSV}")
    df.to_csv(OUTPUT_CSV, index=False)

    print("Done")